# Módulo 1 — Familiarización con la base de datos y filtros sencillos (Polars)

En este notebook exploramos de forma sencilla el dataset **Operación de pasajeros y despacho de vehículos** (~49.2 millones de registros) usando exclusivamente **Polars** en modo *lazy* (`scan_parquet`), lo que permite filtrar y resumir millones de filas en menos de un segundo.

## 1. Carga en modo *lazy*

In [3]:
from datetime import date
from pathlib import Path
import polars as pl

RUTA_PARQUET = Path("../data/raw/operacion_pasajeros.parquet")

# scan_parquet no carga todo en RAM; construye un plan de consulta optimizado
lf = pl.scan_parquet(RUTA_PARQUET)
print(f"Polars {pl.__version__} | Archivo: {RUTA_PARQUET.name} ({RUTA_PARQUET.stat().st_size / 1e6:.1f} MB)")

Polars 1.43.2 | Archivo: operacion_pasajeros.parquet (213.1 MB)


## 2. Estructura, primeras filas y conteo básico

In [2]:
# Columnas y tipos de dato
lf.collect_schema()

Schema([('terminal', String),
        ('clase_vehiculo', String),
        ('nivel_servicio', String),
        ('municipio_origen_ruta', String),
        ('municipio_destino_ruta', String),
        ('fecha_despacho', Date),
        ('hora_despacho', String),
        ('tipo_despacho', String),
        ('despachos', Int32),
        ('pasajeros', Int32)])

In [3]:
# Primeras 10 filas
lf.head(10).collect()

terminal,clase_vehiculo,nivel_servicio,municipio_origen_ruta,municipio_destino_ruta,fecha_despacho,hora_despacho,tipo_despacho,despachos,pasajeros
str,str,str,str,str,date,str,str,i32,i32
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""07""","""ORIGEN""",1,0
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""07""","""ORIGEN""",1,0
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""06""","""ORIGEN""",1,0
"""T.T. DE PASTO""","""MICROBUS""","""BASICO""","""52001""","""52356""",2019-10-04,"""10""","""ORIGEN""",1,5
"""T.T. DE PASTO""","""MICROBUS""","""BASICO""","""52001""","""52356""",2019-10-05,"""05""","""ORIGEN""",1,15
"""T.T. DE PASTO""","""BUS""","""BASICO""","""76001""","""52356""",2019-10-05,"""17""","""TRANSITO""",1,19
"""T.T. DE PASTO""","""MICROBUS""","""BASICO""","""52001""","""52356""",2019-10-05,"""12""","""ORIGEN""",1,15
"""T.T. DE PASTO""","""BUS""","""BASICO""","""52356""","""76001""",2019-10-06,"""10""","""TRANSITO""",1,8
"""T.T. DE PASTO""","""BUS""","""BASICO""","""52356""","""76001""",2019-11-25,"""16""","""TRANSITO""",1,8


In [4]:
# Total de filas y nulos por columna
total_filas = lf.select(pl.len()).collect().item()
print(f"Total de registros: {total_filas:,}")

lf.null_count().collect()

Total de registros: 49,202,812


terminal,clase_vehiculo,nivel_servicio,municipio_origen_ruta,municipio_destino_ruta,fecha_despacho,hora_despacho,tipo_despacho,despachos,pasajeros
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [7]:
df = lf.collect()
duplicados = int(df.is_duplicated().sum())

print(f"Number of duplicate rows: {duplicados}")

Number of duplicate rows: 14681542


## 3. Categorías principales y estadísticas descriptivas

In [ ]:
# Valores únicos de las columnas categóricas pequeñas
for col in ["clase_vehiculo", "nivel_servicio", "tipo_despacho"]:
    valores = lf.select(pl.col(col).unique().sort()).collect()[col].to_list()
    print(f"{col}: {valores}")

clase_vehiculo: ['AUTOMOVIL', 'BUS', 'BUSETA', 'CAMIONETA', 'CAMPERO', 'MICROBUS']


nivel_servicio: ['BASICO', 'LUJO', 'PREFERENCIAL DE LUJO']


tipo_despacho: ['ORIGEN', 'TRANSITO']


In [ ]:
# Estadísticas descriptivas de variables numéricas y fechas
lf.select(["fecha_despacho", "despachos", "pasajeros"]).describe()

statistic,fecha_despacho,despachos,pasajeros
str,str,f64,f64
"""count""","""49202812""",4.9202812e7,4.9202812e7
"""null_count""","""0""",0.0,0.0
"""mean""","""2023-06-29 04:13:03.935682""",1.033256,8.546153
"""std""",null,0.22909,8.500875
"""min""","""2019-08-04""",1.0,0.0
"""25%""","""2022-03-15""",1.0,3.0
"""50%""","""2023-03-20""",1.0,7.0
"""75%""","""2025-03-13""",1.0,12.0
"""max""","""6202-04-13""",20.0,640.0


## 4. Aplicación de filtros sencillos

Con Polars usamos `.filter(...)` sobre el `LazyFrame` y al final `.collect()` para obtener el resultado.

In [ ]:
# Filtro 1: Registros de una terminal específica
salitre = (
    lf.filter(pl.col("terminal") == "T.T. DE BOGOTÁ SALITRE")
    .head(5)
    .collect()
)
salitre

terminal,clase_vehiculo,nivel_servicio,municipio_origen_ruta,municipio_destino_ruta,fecha_despacho,hora_despacho,tipo_despacho,despachos,pasajeros
str,str,str,str,str,date,str,str,i32,i32
"""T.T. DE BOGOTÁ SALITRE""","""MICROBUS""","""LUJO""","""11001""","""41551""",2020-12-10,"""09""","""ORIGEN""",1,8
"""T.T. DE BOGOTÁ SALITRE""","""BUSETA""","""LUJO""","""11001""","""25878""",2020-12-11,"""08""","""ORIGEN""",1,3
"""T.T. DE BOGOTÁ SALITRE""","""BUS""","""LUJO""","""11001""","""68001""",2020-12-15,"""14""","""ORIGEN""",1,13
"""T.T. DE BOGOTÁ SALITRE""","""BUS""","""LUJO""","""11001""","""41001""",2020-12-23,"""22""","""ORIGEN""",1,0
"""T.T. DE BOGOTÁ SALITRE""","""MICROBUS""","""LUJO""","""11001""","""68861""",2020-12-25,"""11""","""ORIGEN""",1,7


In [ ]:
# Filtro 2: Rango de fechas (ej. año 2023 completo)
anio_2023 = lf.filter(
    pl.col("fecha_despacho").is_between(date(2023, 1, 1), date(2023, 12, 31))
)

anio_2023.select([
    pl.len().alias("registros_2023"),
    pl.col("pasajeros").sum().alias("pasajeros_2023"),
    pl.col("despachos").sum().alias("despachos_2023"),
]).collect()

registros_2023,pasajeros_2023,despachos_2023
u32,i32,i32
12306865,108481982,12661033


In [ ]:
# Filtro 3: Múltiples condiciones (& y .is_in)
# Despachos de ORIGEN en BUS o BUSETA con al menos 1 pasajero
filtro_combinado = (
    lf.filter(
        (pl.col("tipo_despacho") == "ORIGEN")
        & pl.col("clase_vehiculo").is_in(["BUS", "BUSETA"])
        & (pl.col("pasajeros") > 0)
    )
    .select(["terminal", "clase_vehiculo", "fecha_despacho", "despachos", "pasajeros"])
    .head(10)
    .collect()
)
filtro_combinado

terminal,clase_vehiculo,fecha_despacho,despachos,pasajeros
str,str,date,i32,i32
"""T.T. DE PASTO""","""BUS""",2019-12-06,1,39
"""T.T. DE PASTO""","""BUS""",2019-12-08,1,40
"""T.T. DE PASTO""","""BUS""",2019-12-13,1,25
"""T.T. DE PASTO""","""BUS""",2019-12-15,1,16
"""T.T. DE PASTO""","""BUS""",2019-12-17,1,40
"""T.T. DE PASTO""","""BUS""",2019-12-20,1,18
"""T.T. DE PASTO""","""BUS""",2019-12-20,1,37
"""T.T. DE PASTO""","""BUS""",2019-12-21,1,40
"""T.T. DE PASTO""","""BUS""",2019-12-21,2,80


In [ ]:
# Filtro 4: Una ruta específica por códigos DIVIPOLA
# Ejemplo: Medellín (05001) -> Bogotá (11001)
med_bog = lf.filter(
    (pl.col("municipio_origen_ruta") == "05001")
    & (pl.col("municipio_destino_ruta") == "11001")
)

med_bog.select([
    pl.len().alias("registros_ruta"),
    pl.col("pasajeros").sum().alias("total_pasajeros"),
    pl.col("pasajeros").mean().round(2).alias("promedio_pasajeros_por_registro"),
]).collect()

registros_ruta,total_pasajeros,promedio_pasajeros_por_registro
u32,i32,f64
72157,1597047,22.13


## 5. Resúmenes sencillos (agrupaciones)

In [ ]:
# Top 10 terminales por volumen histórico de pasajeros
(
    lf.group_by("terminal")
    .agg([
        pl.len().alias("registros"),
        pl.col("despachos").sum().alias("total_despachos"),
        pl.col("pasajeros").sum().alias("total_pasajeros"),
    ])
    .sort("total_pasajeros", descending=True)
    .head(10)
    .collect()
)

terminal,registros,total_despachos,total_pasajeros
str,u32,i32,i32
"""T.T. DE CALI""",4012777,4212747,36866048
"""T.T. DE MEDELLÍN NORTE""",2748727,2774836,36620730
"""T.T. DE BOGOTÁ SALITRE""",4501536,4558984,33530397
"""T.T. DE PEREIRA""",2463278,2553053,20095271
"""T.T. DE ARMENIA""",2669142,2704701,18107755
"""T.T. DE TULÚA""",1305139,1325533,13505557
"""T.T. DE NEIVA""",1022713,1025222,12953910
"""T.T. DE IBAGUÉ""",1143593,1154308,12823896
"""T.T. DE TUNJA""",1568740,1590984,12534643


In [ ]:
# Resumen anual (filtrando los 3 registros con años > 2026 de origen)
(
    lf.filter(pl.col("fecha_despacho") <= date(2026, 12, 31))
    .with_columns(pl.col("fecha_despacho").dt.year().alias("anio"))
    .group_by("anio")
    .agg([
        pl.len().alias("registros"),
        pl.col("despachos").sum().alias("despachos"),
        pl.col("pasajeros").sum().alias("pasajeros"),
    ])
    .sort("anio")
    .collect()
)

anio,registros,despachos,pasajeros
i32,u32,i32,i32
2019,77,79,1077
2020,14265,20982,147042
2021,9959766,10267496,67566275
2022,12018845,12438576,101341991
2023,12306865,12661033,108481982
2024,106498,124417,1135756
2025,11667035,12107729,111418527
2026,3129458,3218785,30402103


In [ ]:
# Ejemplo de serie diaria para una ruta (Medellín -> Bogotá en enero de 2023)
(
    lf.filter(
        (pl.col("municipio_origen_ruta") == "05001")
        & (pl.col("municipio_destino_ruta") == "11001")
        & pl.col("fecha_despacho").is_between(date(2023, 1, 1), date(2023, 1, 31))
    )
    .group_by("fecha_despacho")
    .agg([
        pl.col("despachos").sum().alias("despachos"),
        pl.col("pasajeros").sum().alias("pasajeros"),
    ])
    .sort("fecha_despacho")
    .head(10)
    .collect()
)

fecha_despacho,despachos,pasajeros
date,i32,i32
2023-01-01,31,881
2023-01-02,40,1496
2023-01-03,52,2004
2023-01-04,49,1959
2023-01-05,50,1832
2023-01-06,54,2007
2023-01-07,53,1830
2023-01-08,48,1901
2023-01-09,49,1811


In [4]:
#datos por año
df = lf.collect()
año = df.with_columns(pl.col("fecha_despacho").dt.year().alias("anio"))


In [5]:
año.head(100)

terminal,clase_vehiculo,nivel_servicio,municipio_origen_ruta,municipio_destino_ruta,fecha_despacho,hora_despacho,tipo_despacho,despachos,pasajeros,anio
str,str,str,str,str,date,str,str,i32,i32,i32
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""07""","""ORIGEN""",1,0,2019
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""07""","""ORIGEN""",1,0,2019
"""T.T. DE SAN GIL""","""BUSETA""","""BASICO""","""68679""","""68755""",2019-08-04,"""06""","""ORIGEN""",1,0,2019
"""T.T. DE PASTO""","""MICROBUS""","""BASICO""","""52001""","""52356""",2019-10-04,"""10""","""ORIGEN""",1,5,2019
"""T.T. DE PASTO""","""MICROBUS""","""BASICO""","""52001""","""52356""",2019-10-05,"""05""","""ORIGEN""",1,15,2019
…,…,…,…,…,…,…,…,…,…,…
"""T.T. DE POPAYÁN""","""CAMIONETA""","""BASICO""","""19001""","""19256""",2020-01-05,"""10""","""ORIGEN""",1,6,2020
"""T.T. DE POPAYÁN""","""MICROBUS""","""BASICO""","""19001""","""19450""",2020-01-05,"""10""","""ORIGEN""",1,12,2020
"""T.T. DE POPAYÁN""","""MICROBUS""","""BASICO""","""76001""","""41551""",2020-01-05,"""09""","""TRANSITO""",1,12,2020


In [13]:
# número de registros por año
año["anio"].value_counts().sort("anio")

anio,count
i32,u32
2019,77
2020,14265
2021,9959766
2022,12018845
2023,12306865
…,…
2025,11667035
2026,3129458
2202,1


In [14]:
df["despachos"].unique().sort()

despachos
i32
1
2
3
4
5
…
16
17
18
